In [8]:
%pip install matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ----------------------------- ---------- 6.8/9.3 MB 38.1 MB/s eta 0:00:01
   ---------------------------------------  9.2/9.3 MB 28.5 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 22.3 MB/s  0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 22.9 MB/s  0:00:00
   ---------------------------------------- 0.0/7.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.3 MB ? eta -:--:--
   ------------------------------ --------- 5.5/7.3 MB 22.4 MB/s eta 0:00:01
   ---------------------------------------- 7.3/7.3 MB 21.4 MB/s  0:00:00

   ---------------------------------------- 0/7 [pyparsing]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [9]:
import numpy as np
import noise_functions 
import synthetic_functions 

In [10]:
import numpy as np
import matplotlib.pyplot as plt

M = 100
N = 1000
r = 10

eps_values = [1, 2, 5]
num_trials = 1000

# Generate synthetic data
X, Q = synthetic_functions.generate_synthetic_data(M, N, r)

# Original matrix
R = X @ X.T

# Coordinate-wise sensitivity bounds
lambda_vec = synthetic_functions.generate_lambda(M)

# Clipped matrix for i.n.i.d.
R_clipped = synthetic_functions.clip_R(X, lambda_vec)

# Store results
iid_results = []
inid_results = []

for epsilon in eps_values:

    iid_errors = []
    inid_errors = []

    for trial in range(num_trials):

        # I.I.D. Laplace
        R_iid = noise_functions.laplace_noise_iid(
            R,
            epsilon,
            lambda_vec
        )

        # I.N.I.D. Laplace
        R_inid = noise_functions.laplace_noise_inid(
            R_clipped,
            epsilon,
            lambda_vec
        )

        # Calculate errors
        iid_error = synthetic_functions.subspace_recovery_error(
            R, R_iid, r
        )

        inid_error = synthetic_functions.subspace_recovery_error(
            R, R_inid, r
        )

        iid_errors.append(iid_error)
        inid_errors.append(inid_error)

    # Average across trials
    iid_mean = np.mean(iid_errors)
    inid_mean = np.mean(inid_errors)

    iid_results.append(iid_mean)
    inid_results.append(inid_mean)

    print(f"Epsilon = {epsilon}")
    print(f"I.I.D. Laplace Average SRE: {iid_mean:.4f}")
    print(f"I.N.I.D. Laplace Average SRE: {inid_mean:.4f}")
    print()

Epsilon = 1
I.I.D. Laplace Average SRE: 0.9370
I.N.I.D. Laplace Average SRE: 0.9429

Epsilon = 2
I.I.D. Laplace Average SRE: 0.9218
I.N.I.D. Laplace Average SRE: 0.9357

Epsilon = 5
I.I.D. Laplace Average SRE: 0.8541
I.N.I.D. Laplace Average SRE: 0.9098



In [11]:
print("Lambda L1 norm:", np.linalg.norm(lambda_vec, 1))

beta = (
    np.sum(lambda_vec**(2/3))
    * lambda_vec**(1/3)
    / epsilon
)

print("Minimum beta:", np.min(beta))
print("Maximum beta:", np.max(beta))

print("Largest eigenvalue of R:")
print(np.linalg.eigvalsh(R)[-1])

print("Largest eigenvalue of R_clipped:")
print(np.linalg.eigvalsh(R_clipped)[-1])

Lambda L1 norm: 50.66153600485505
Minimum beta: 10.101601843353656
Maximum beta: 11.338664696296481
Largest eigenvalue of R:
113.01042380295631
Largest eigenvalue of R_clipped:
65.54414859914279
